In [60]:
from sqlalchemy import create_engine
import pandas as pd
import yaml
import os

filename = os.path.join(os.getcwd(), 'creds.yaml')


with open(filename, "r") \
      as file:
    creds = yaml.safe_load(file)


def reads_from_mysql(creds, query):
    _db_user = creds['username']
    _db_password = creds['password']
    _db_host = creds['host']
    _db_name = creds['database']
    engine = create_engine(f"mysql://{_db_user}:{_db_password}@{_db_host}:3306/{_db_name}")
    df = pd.read_sql(query, engine)
    return df

def write_to_database(creds, df, table_name, if_exists='append'):
    """
    Returns as dataframe the result of a query to a MySQL database.

    Args:
        creds (dict): The credentials to access the database.
        query (string): The query.

    Returns:
        pandas dataframe: the output table of the query.
    """
    _db_user = creds['username']
    _db_password = creds['password']
    _db_host = creds['host']
    _db_name = creds['database']
    engine = create_engine(f"mysql://{_db_user}:{_db_password}@{_db_host}:3306/{_db_name}")
    with engine.connect() as connection:
        df.to_sql(table_name, con=connection, if_exists=if_exists, index=False) 


In [61]:
import pandas as pd

In [62]:
df = pd.read_csv('C:/Users/massi/OneDrive/Desktop/Uni/DMFBI-R/4.nocode/data/invoices_eae.csv',sep=';')

In [63]:
write_to_database(creds=creds['mysql-db'], df=df, table_name='python_test')

ASSIGNMENT

In [64]:
meteo_types = {'temperature':'float64','relative_humidity':'float64','precipitation_rate':'float64','wind_speed':'float64','zipcode':'str'}
contracts_types = {'CONTRACT_ID':'int64','CLIENT_TYPE_ID':'int64','AVG_EUROS_IMPORT':'float64','POWER_P1':'float64','HAS_GAS':'boolean','HAS_SOLAR':'boolean','ZIPCODE':'str'}
zipcode_types = {'ZIPCODE':'str','ZC_LATITUDE':'float64','ZC_LONGITUDE':'float64','AUTONOMOUS_COMMUNITY':'str','AUTONOMOUS_COMMUNITY_NK':'str','PROVINCE':'str'}

In [65]:
def _filter_data_isin(table: str, column: str, lookup: list):
    '''
    Arg:
        table -> table you want to filter
        column -> table column as input for the lookup
        lookup -> list or any iterable that contains lookup values
    '''
    return table[table[column].isin(lookup)]
df_contracts = pd.read_csv('contracts_eae.csv', dtype=contracts_types)
df_zipcode = pd.read_csv('zipcode_eae_v2.csv', dtype=zipcode_types)

In [66]:
df_contracts.columns = df_contracts.columns.str.lower()
df_zipcode.columns = df_zipcode.columns.str.lower()

In [67]:
zipcode_top =  list(df_contracts.groupby('zipcode')['contract_id'].count().reset_index().nlargest(10,'contract_id')['zipcode'])

In [68]:
chunks = pd.read_csv('meteo_eae.csv', chunksize = 100000, delimiter=';', \
                        dtype = meteo_types, parse_dates= ['date'])
df_meteo_top = pd.concat([_filter_data_isin(table=chunk, \
                            column='zipcode',lookup=zipcode_top) \
                            for chunk in chunks], ignore_index=True)


In [69]:
def _category_p(power: float) -> str:
    if power >= 5000:
        return 'Over 5 MW'
    elif power < 3000:
        return 'Under 3 MW'
    else:
        return 'Between 3 and 5 MW'
df_contracts['p1_category'] = df_contracts['power_p1'].apply(lambda x: _category_p(x))
df_contracts['p1_category'].unique()

array(['Over 5 MW', 'Between 3 and 5 MW', 'Under 3 MW'], dtype=object)

In [70]:
df_contracts_zero = df_contracts[df_contracts['client_type_id']==0]

In [71]:
df_contracts_temperatures = df_contracts_zero.merge(df_meteo_top, how='right', left_on='zipcode', right_on='zipcode')